<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 24 · Backpropagation tự cài</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Lan truyền ngược bằng tay trên một nơ-ron, công thức cho cả một lớp, rồi tự cài mạng hai lớp bằng NumPy, kiểm gradient và huấn luyện trên hai vầng trăng.</p>
</div>

### Sau notebook này bạn sẽ

- Vẽ đồ thị tính toán và lan truyền ngược bằng tay.
- Dẫn dW = Xᵀ·dZ và đạo hàm ReLU dạng mặt nạ.
- Tự cài mạng hai lớp và kiểm gradient bằng sai phân.
- Huấn luyện mạng tự viết tới 93% trên hai vầng trăng.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/01-do-thi-tinh-toan/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Đồ thị tính toán](#trang-1)
2. [Trang 2 · Lan truyền ngược qua một lớp](#trang-2)
3. [Trang 3 · Tự cài mạng hai lớp](#trang-3)
4. [Trang 4 · Huấn luyện mạng tự cài](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Đồ thị tính toán

Một nơ-ron: lượt xuôi, lượt ngược, gradient check.

*Đọc kèm trên web: [Trang 1 · Đồ thị tính toán](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/01-do-thi-tinh-toan/)*

In [ ]:
import numpy as np                                  # tính toán
x, w, b, y = 2.0, 0.5, 0.1, 1.0                     # đầu vào, tham số, nhãn
z = w * x + b                                       # lượt xuôi
a = 1 / (1 + np.exp(-z))                            # sigmoid
L = (a - y) ** 2                                    # loss bình phương
dL_da = 2 * (a - y)                                 # lượt ngược bắt đầu từ loss
dL_dz = dL_da * a * (1 - a)                         # qua sigmoid
dL_dw, dL_db = dL_dz * x, dL_dz                     # qua phép nhân và phép cộng
print(round(z, 4), round(a, 4), round(L, 4), round(dL_dz, 4), round(dL_dw, 4), round(dL_db, 4))   # giá trị và gradient
h = 1e-5                                            # bước sai phân
f = lambda w_: (1 / (1 + np.exp(-(w_ * x + b))) - y) ** 2     # loss theo w
print(round((f(w + h) - f(w - h)) / (2 * h), 4))    # gradient check

Output mong đợi:

```text
1.1 0.7503 0.0624 -0.0936 -0.1872 -0.0936
-0.1872
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Lan truyền ngược qua một lớp

Đạo hàm của ReLU là một mặt nạ.

*Đọc kèm trên web: [Trang 2 · Lan truyền ngược qua một lớp](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/02-lan-truyen-qua-mot-lop/)*

In [ ]:
Z = np.array([[1.5, -0.3], [-2.0, 0.7]])     # đầu vào của ReLU
dA = np.array([[0.2, 0.4], [0.1, -0.5]])     # gradient từ lớp sau
print((Z > 0).astype(int))                   # mặt nạ 0 và 1
print(dA * (Z > 0))                          # công thức (24.2)

Output mong đợi:

```text
[[1 0]
 [0 1]]
[[ 0.2  0. ]
 [ 0.  -0.5]]
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Tự cài mạng hai lớp

Mạng 2-32-1 bằng NumPy và gradient check.

*Đọc kèm trên web: [Trang 3 · Tự cài mạng hai lớp](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/03-tu-cai-mang-hai-lop/)*

In [ ]:
from sklearn.datasets import make_moons                     # dữ liệu
from sklearn.model_selection import train_test_split        # chia
Xm, ym = make_moons(300, noise=0.3, random_state=0)         # hai vầng trăng
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=0)   # 210 train, 90 test
r = np.random.default_rng(0)                                # seed
P = [r.normal(0, 1.0, (2, 32)),                             # W1, khởi tạo He với d = 2
     np.zeros(32),                                          # b1
     r.normal(0, np.sqrt(2 / 32), (32, 1)),                 # W2, khởi tạo He với d = 32
     np.zeros(1)]                                           # b2
print([p.shape for p in P], sum(p.size for p in P))         # shape và số tham số

def truyen_xuoi(P, X):                                      # lượt xuôi
    W1, b1, W2, b2 = P                                      # mở gói tham số
    Z1 = X @ W1 + b1                                        # (n, 32)
    A1 = np.maximum(0, Z1)                                  # ReLU
    p = 1 / (1 + np.exp(-(A1 @ W2 + b2)))                   # sigmoid ở lớp ra
    return Z1, A1, p.ravel()                                # giữ Z1, A1 cho lượt ngược

def mat_mat(P, X, y):                                       # cross-entropy
    p = np.clip(truyen_xuoi(P, X)[2], 1e-12, 1 - 1e-12)     # tránh log(0)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))   # trung bình

def truyen_nguoc(P, X, y):                                  # lượt ngược
    W1, b1, W2, b2 = P                                      # mở gói
    Z1, A1, p = truyen_xuoi(P, X)                           # cần giá trị lượt xuôi
    dZ2 = (p - y)[:, None] / len(y)                         # (24.3)
    dW2, db2 = A1.T @ dZ2, dZ2.sum(0)                       # (24.1) lớp ra
    dA1 = dZ2 @ W2.T                                        # gửi về lớp ẩn
    dZ1 = dA1 * (Z1 > 0)                                    # (24.2) qua ReLU
    dW1, db1 = X.T @ dZ1, dZ1.sum(0)                        # (24.1) lớp ẩn
    return [dW1, db1, dW2, db2]                             # cùng thứ tự với P

Output mong đợi:

```text
[(2, 32), (32,), (32, 1), (1,)] 129
```

In [ ]:
G = truyen_nguoc(P, Xtr, ytr)                            # gradient bằng công thức
for k, i in [(0, (0, 0)), (2, (5, 0)), (1, (3,))]:       # ba tham số bất kỳ
    cu = P[k][i]                                         # giữ giá trị cũ
    P[k][i] = cu + h; lp = mat_mat(P, Xtr, ytr)          # nhích lên
    P[k][i] = cu - h; lm = mat_mat(P, Xtr, ytr)          # nhích xuống
    P[k][i] = cu                                         # trả lại
    print(k, i, round(float(G[k][i]), 8), round((lp - lm) / (2 * h), 8))   # công thức và sai phân

Output mong đợi:

```text
0 (0, 0) -0.01272586 -0.01272586
2 (5, 0) -0.05968156 -0.05968156
1 (3,) 0.09515478 0.09515478
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI KIỂM TRƯỚC KHI HUẤN LUYỆN?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Gradient sai không làm chương trình báo lỗi: loss vẫn có thể giảm, chỉ chậm hoặc dừng sai chỗ. Kiểm trước khi huấn luyện.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Huấn luyện mạng tự cài

3000 bước gradient descent.

*Đọc kèm trên web: [Trang 4 · Huấn luyện mạng tự cài](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/04-huan-luyen/)*

In [ ]:
dung = lambda X, y: float(np.mean((truyen_xuoi(P, X)[2] > 0.5) == y))   # độ chính xác
for buoc in range(3001):                                                # 3000 bước
    if buoc in (0, 100, 1000, 3000):                                    # in vài mốc
        print(buoc, round(mat_mat(P, Xtr, ytr), 4), round(dung(Xtr, ytr), 4), round(dung(Xte, yte), 4))   # loss, train, test
    G = truyen_nguoc(P, Xtr, ytr)                                       # gradient
    P = [p - 0.5 * g for p, g in zip(P, G)]                             # cập nhật, learning rate 0,5

Output mong đợi:

```text
0 1.1723 0.2095 0.3111
100 0.312 0.8476 0.8667
1000 0.1673 0.9381 0.9444
3000 0.1474 0.9429 0.9333
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Đổi learning rate thành 5. Loss có thành nan không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đổi 32 nơ-ron thành 4. Test còn được bao nhiêu?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch24-backpropagation/05-bai-tap/)*

### Bài 24.1 · Nhãn đổi thành 0

Với nhãn $y = 0$, tính $\partial L/\partial a$, $\partial L/\partial z$, $\partial L/\partial w$.
<details><summary>Xem đáp án</summary>

$1{,}5005$; $0{,}2812$; $0{,}5623$. Gradient đổi dấu và lớn gấp ba vì đầu ra 0,75 giờ quá cao. Code ở ô ngay dưới.
</details>

In [ ]:
y0 = 0.0                                     # nhãn mới
dz0 = 2 * (a - y0) * a * (1 - a)             # qua loss và sigmoid
print(round(2 * (a - y0), 4), round(dz0, 4), round(dz0 * x, 4))   # dL/da, dL/dz, dL/dw

Output mong đợi:

```text
1.5005 0.2812 0.5623
```

### Bài 24.2 · Shape của mọi gradient

Với mạng 784-128-10 và lô 64 ảnh, viết shape của mọi đại lượng ở lượt xuôi và lượt ngược.
<details><summary>Xem đáp án</summary>

$dW_1$ là (784, 128), $dW_2$ là (128, 10), $Z_1$ và $dA_1$ là (64, 128), $dZ_2$ là (64, 10). Gradient của tham số không phụ thuộc cỡ lô.
</details>

### Bài 24.3 · Khởi tạo đối xứng

Khởi tạo mọi trọng số bằng 0,1, huấn luyện 3000 bước, đếm số giá trị khác nhau trong mỗi hàng của $W_1$.
<details><summary>Xem đáp án</summary>

Mỗi hàng chỉ một giá trị: 32 nơ-ron vẫn giống hệt nhau, test chỉ 83%. Code ở ô ngay dưới.
</details>

In [ ]:
Pd = [np.full((2, 32), 0.1), np.zeros(32), np.full((32, 1), 0.1), np.zeros(1)]   # mọi trọng số bằng nhau
for _ in range(3000):                                                           # huấn luyện như trang 4
    Pd = [p - 0.5 * g for p, g in zip(Pd, truyen_nguoc(Pd, Xtr, ytr))]          # gradient descent
print(np.unique(Pd[0][0].round(6)).size, np.unique(Pd[0][1].round(6)).size, round(float(np.mean((truyen_xuoi(Pd, Xte)[2] > 0.5) == yte)), 4))   # số giá trị khác nhau, test

Output mong đợi:

```text
1 1 0.8333
```